# 🤖 QA IA Angent
Este notebook genera matrices de pruebas y casos Gherkin usando la tecnica de prompt engineering




In [5]:
pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [6]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display
import pandas as pd

load_dotenv()
client = OpenAI()  # Lee OPENAI_API_KEY desde .env

In [7]:
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("❌ No se encontró la variable OPENAI_API_KEY en el archivo .env")
else:
    print("✅ OPENAI_API_KEY cargada correctamente")

✅ OPENAI_API_KEY cargada correctamente


In [8]:
client = OpenAI()

In [9]:
# 3. Requerimiento de entrada (ejemplo real o genérico)
requerimiento = """
Titulo: Login en aplicación móvil

Objetivo: Validar que los usuarios puedan acceder correctamente con credenciales válidas 
y que se bloquee el acceso cuando son inválidas.

Criterios de aceptación:
- Usuario válido con credenciales correctas debe acceder.
- Usuario inválido debe recibir mensaje de error.
- Usuario bloqueado después de 3 intentos fallidos.
"""

In [10]:
# 4. Ejemplo few-shot (enseñar al modelo el formato esperado, genérico)
few_shot_example = """
Ejemplo de matriz de prueba:

| ID    | Escenario                  | Datos de entrada        | Resultado esperado               |
|-------|----------------------------|-------------------------|----------------------------------|
| TC001 | Login exitoso              | Usuario válido          | Acceso concedido                 |
| TC002 | Login fallido              | Usuario inválido        | Mensaje de error                 |
| TC003 | Validación de contraseña   | Contraseña incorrecta   | Bloqueo después de 3 intentos    |

Ejemplo de casos Gherkin:

Feature: Validar login de usuarios

  Scenario: Usuario válido accede con credenciales correctas
    Given un usuario registrado
    When introduce usuario y contraseña válidos
    Then debe acceder exitosamente al sistema

  Scenario: Usuario inválido intenta acceder
    Given un usuario no registrado
    When introduce credenciales inválidas
    Then debe mostrarse un mensaje de error
"""

In [11]:
# 5. Prompt estructurado usando system + user
resp = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": "Eres un QA Engineer experto en generar matrices de prueba y casos Gherkin (Cucumber)."
        },
        {
            "role": "user",
            "content": f"""Aquí tienes un ejemplo del formato esperado:

{few_shot_example}

Ahora, genera la **matriz de pruebas** y los **casos Gherkin** para el siguiente requerimiento:

{requerimiento}
"""
        }
    ],
    temperature=0.3
)

output = resp.choices[0].message.content
print("=== SALIDA IA ===")
print(output)

=== SALIDA IA ===
### Matriz de Pruebas

| ID    | Escenario                  | Datos de entrada                  | Resultado esperado                     |
|-------|----------------------------|-----------------------------------|----------------------------------------|
| TC001 | Login exitoso              | Usuario válido y contraseña correcta | Acceso concedido                       |
| TC002 | Login fallido              | Usuario inválido y contraseña incorrecta | Mensaje de error "Credenciales inválidas" |
| TC003 | Bloqueo por intentos fallidos | Usuario válido y contraseña incorrecta (3 intentos) | Usuario bloqueado, mensaje "Acceso bloqueado" |
| TC004 | Intento después de bloqueo | Usuario válido y contraseña correcta (después de 3 intentos fallidos) | Mensaje "Acceso bloqueado"            |

### Casos Gherkin

Feature: Validar login en aplicación móvil

  Scenario: Usuario válido accede con credenciales correctas
    Given un usuario registrado con credenciales válidas
    W

In [12]:
# 6. Exportar resultados: matriz a Excel y casos a archivo .feature

rows = []
for line in output.splitlines():
    if "|" in line and "---" not in line:
        cols = [c.strip() for c in line.split("|") if c.strip()]
        if len(cols) > 1:
            rows.append(cols)

if rows:
    df = pd.DataFrame(rows[1:], columns=rows[0])
    df.to_excel("matriz_pruebas.xlsx", index=False)
    print("✅ matriz_pruebas.xlsx generado")

gherkin = []
capture = False
for line in output.splitlines():
    if line.strip().startswith(("Feature", "Scenario")):
        capture = True
    if capture:
        gherkin.append(line)

if gherkin:
    with open("casos.feature", "w", encoding="utf-8") as f:
        f.write("\n".join(gherkin))
    print("✅ casos.feature generado")

✅ matriz_pruebas.xlsx generado
✅ casos.feature generado


In [13]:
import pandas as pd

df = pd.read_excel("matriz_pruebas.xlsx")
df

,ID,Escenario,Datos de entrada,Resultado esperado
0,TC001,Login exitoso,Usuario válido y contraseña correcta,Acceso concedido
1,TC002,Login fallido,Usuario inválido y contraseña incorrecta,"Mensaje de error ""Credenciales inválidas"""
2,TC003,Bloqueo por intentos fallidos,Usuario válido y contraseña incorrecta (3 inte...,"Usuario bloqueado, mensaje ""Acceso bloqueado"""
3,TC004,Intento después de bloqueo,Usuario válido y contraseña correcta (después ...,"Mensaje ""Acceso bloqueado"""


In [14]:
# 7. Leer y mostrar la matriz de pruebas generada en el notebook
import pandas as pd

# Cargar el Excel generado
df = pd.read_excel("matriz_pruebas.xlsx")

# Mostrarlo como tabla en Jupyter
df

,ID,Escenario,Datos de entrada,Resultado esperado
0,TC001,Login exitoso,Usuario válido y contraseña correcta,Acceso concedido
1,TC002,Login fallido,Usuario inválido y contraseña incorrecta,"Mensaje de error ""Credenciales inválidas"""
2,TC003,Bloqueo por intentos fallidos,Usuario válido y contraseña incorrecta (3 inte...,"Usuario bloqueado, mensaje ""Acceso bloqueado"""
3,TC004,Intento después de bloqueo,Usuario válido y contraseña correcta (después ...,"Mensaje ""Acceso bloqueado"""
